# Temporal Fusion Transformer (TFT) — Walmart Store Sales Forecasting
Pipeline: data download, preprocessing, feature engineering (with proper future/static covariates for TFT), hyperparameter search, final fit, and MLflow/W&B logging.

In [1]:
!pip install -q kaggle wandb dagshub mlflow "neuralforecast>=1.7.4" pandas numpy scikit-learn pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14

In [2]:
import os

def get_secret(name: str):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)


In [3]:
import os
import glob


os.environ["KAGGLE_API_TOKEN"] = 'KGAT_a3a9b1a151af12c2215e74a07293899d'

DATA_DIR = "/content/data" if os.path.isdir("/content") else "./data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p {DATA_DIR}
!unzip -oq {DATA_DIR}/walmart-recruiting-store-sales-forecasting.zip -d {DATA_DIR}

for z in glob.glob(f"{DATA_DIR}/*.csv.zip"):
    get_ipython().system(f'unzip -oq "{z}" -d {DATA_DIR}')

print(sorted(os.listdir(DATA_DIR)))

100% 2.70M/2.70M [00:00<00:00, 217MB/s]

['features.csv', 'features.csv.zip', 'sampleSubmission.csv', 'sampleSubmission.csv.zip', 'stores.csv', 'test.csv', 'test.csv.zip', 'train.csv', 'train.csv.zip', 'walmart-recruiting-store-sales-forecasting.zip']


In [4]:
import wandb

wandb_key = get_secret("WANDB_API_KEY")
assert wandb_key, "Set WANDB_API_KEY as a secret before running this cell."
wandb.login(key=wandb_key)

WANDB_PROJECT = "ml-final-projekt-walmart-sales-forecasting"
WANDB_RUN_NAME = "tft-walmart"

import dagshub
import mlflow

dagshub.init(
    repo_owner="lshek22",
    repo_name="walmart-recruiting-store-sales-forecasting",
    mlflow=True,
)
mlflow.set_experiment("TFT_Training")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lshek22 (ml-final-projekt) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=18378a3c-edd0-4e9a-94d9-49e8a40a23c1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5f87000a90910219c356a7b3db84b2dcee728a5fd8f4c9ce2d028d896c4f9cea




Accessing as lshek22

Initialized MLflow to track repo "lshek22/walmart-recruiting-store-sales-forecasting"

Repository lshek22/walmart-recruiting-store-sales-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/636dfb3e5184486db89edc996bb32a5a', creation_time=1783519791479, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783519791479, lifecycle_stage='active', name='TFT_Training', tags={}, trace_location=None, workspace='default'>

In [5]:
import os
# Must be set before the first `import torch` — reduces CUDA memory fragmentation,
# which is the main reason repeated fit() calls in a sweep loop run out of memory
# even though a single run fits comfortably.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from neuralforecast.models import TFT
from neuralforecast import NeuralForecast

import torch
print(torch.cuda.is_available())


train = pd.read_csv(f"{DATA_DIR}/train.csv.zip", parse_dates=["Date"])
test = pd.read_csv(f"{DATA_DIR}/test.csv.zip", parse_dates=["Date"])
features = pd.read_csv(f"{DATA_DIR}/features.csv.zip", parse_dates=["Date"])
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")


print(train.shape, test.shape, features.shape, stores.shape)
train.head()


True
(421570, 5) (115064, 4) (8190, 12) (45, 3)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [6]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, stores, features):
        self.stores = stores
        self.features = features
        self.label_encoder = None

    def fit(self, X, y=None):
        all_data = X.merge(self.stores, how='left', on='Store')
        if 'Type' in all_data.columns:
            self.label_encoder = LabelEncoder()
            self.label_encoder.fit(all_data['Type'].astype(str))
        return self

    def transform(self, X):
        X = X.copy()
        X = X.merge(self.stores, how='left', on='Store')
        X = X.merge(self.features, how='left', on=['Store', 'Date', 'IsHoliday'])

        X['Date'] = pd.to_datetime(X['Date'])

        if self.label_encoder is not None and 'Type' in X.columns:
            X['Type'] = self.label_encoder.transform(X['Type'].astype(str))

        X.fillna(0, inplace=True)

        X.sort_values(['Store', 'Dept', 'Date'], inplace=True)
        X.reset_index(drop=True, inplace=True)

        return X

df = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')
features = pd.read_csv(f'{DATA_DIR}/features.csv.zip')

preprocessor = Preprocessor(stores=stores, features=features)
preprocessor.fit(df)
df = preprocessor.transform(df)

df['Weekly_Sales'] = df['Weekly_Sales'].astype(float)

val_date = pd.to_datetime('2011-10-01')

train_df = df[df['Date'] < val_date]
valid_df = df[df['Date'] >= val_date]

X_train = train_df.drop(columns=['Weekly_Sales'])
y_train = train_df['Weekly_Sales']

X_valid = valid_df.drop(columns=['Weekly_Sales'])
y_valid = valid_df['Weekly_Sales']


## `NeuralForecastWrapper` with static / historical / future covariates
The previous version silently dropped every engineered column — `_prepare_df` only kept `['unique_id', 'ds', 'y']`, so `FeatureEngineer`'s output never reached the model. This version threads three separate column lists through to `neuralforecast`:

- **`futr_exog_list`** — columns known in advance for the forecast horizon (calendar features, and the weather/economic/markdown columns from `features.csv`, which Walmart provides through the end of the test period). Passed to the model at both train and predict time via `futr_df`.
- **`hist_exog_list`** — columns only available for observed history (e.g. a lag feature like last year's sales, which isn't fully known across the whole 60-week horizon since `h > 52`). Used only inside the encoder window, never extrapolated into the future.
- **`stat_exog_list`** — one row per `unique_id`, doesn't vary over time (Store, Dept, Type, Size). Standardized with a `StandardScaler` fit on train, since `neuralforecast`'s built-in `scaler_type` only normalizes temporal inputs, not static ones.

In [7]:
from sklearn.preprocessing import StandardScaler

class NeuralForecastWrapper:
    def __init__(self, model, freq='W-FRI', group_cols=['Store', 'Dept'], date_col='Date',
                 hist_exog_list=None, futr_exog_list=None, stat_exog_list=None):
        self.model = model
        self.freq = freq
        self.group_cols = group_cols
        self.date_col = date_col
        self.hist_exog_list = list(hist_exog_list) if hist_exog_list else []
        self.futr_exog_list = list(futr_exog_list) if futr_exog_list else []
        self.stat_exog_list = list(stat_exog_list) if stat_exog_list else []
        self.stat_scaler = StandardScaler() if self.stat_exog_list else None
        self.nf = None
        self.fitted = False

    def _make_unique_id(self, X):
        return X[self.group_cols].astype(str).agg('-'.join, axis=1)

    def _prepare_df(self, X, y=None):
        df = X.copy()
        df['ds'] = pd.to_datetime(df[self.date_col])
        df['unique_id'] = self._make_unique_id(df)
        cols = ['unique_id', 'ds'] + self.hist_exog_list + self.futr_exog_list
        if y is not None:
            df['y'] = y.values
            cols = cols + ['y']
        return df[cols]

    def _prepare_futr_df(self, X):
        df = X.copy()
        df['ds'] = pd.to_datetime(df[self.date_col])
        df['unique_id'] = self._make_unique_id(df)
        return df[['unique_id', 'ds'] + self.futr_exog_list]

    def _prepare_static_df(self, X, fit=False):
        if not self.stat_exog_list:
            return None
        df = X.copy()
        df['unique_id'] = self._make_unique_id(df)
        static_df = (
            df[['unique_id'] + self.stat_exog_list]
            .drop_duplicates(subset='unique_id')
            .reset_index(drop=True)
        )
        if fit:
            static_df[self.stat_exog_list] = self.stat_scaler.fit_transform(static_df[self.stat_exog_list])
        else:
            static_df[self.stat_exog_list] = self.stat_scaler.transform(static_df[self.stat_exog_list])
        return static_df

    def fit(self, X, y):
        df = self._prepare_df(X, y)
        static_df = self._prepare_static_df(X, fit=True)
        self.nf = NeuralForecast(models=[self.model], freq=self.freq)
        self.nf.fit(df=df, static_df=static_df)
        self.fitted = True

    def predict(self, X_test):
        if not self.fitted:
            raise RuntimeError("Call fit() before predict().")

        test_df = self._prepare_df(X_test)[['unique_id', 'ds']]
        test_df['ds'] = pd.to_datetime(test_df['ds'])

        futr_df = None
        if self.futr_exog_list:
            # `X_test` isn't guaranteed to contain exactly the (unique_id, ds) combinations
            # the model expects for its h-step horizon (some Store-Dept series have gaps, or
            # X_test may cover a different date range) — building futr_df directly from X_test
            # causes `nf.predict()` to raise "missing combinations of ids and times in futr_df".
            # `make_future_dataframe()` returns the exact skeleton the fitted model needs, and
            # exogenous values are left-merged onto it; any combination absent from X_test
            # (e.g. a series with no recorded row for that future week) falls back to 0.
            exog_values = self._prepare_futr_df(X_test)
            exog_values['ds'] = pd.to_datetime(exog_values['ds'])

            futr_df = self.nf.make_future_dataframe()
            futr_df['ds'] = pd.to_datetime(futr_df['ds'])
            futr_df = futr_df.merge(exog_values, on=['unique_id', 'ds'], how='left')

            n_missing = futr_df[self.futr_exog_list].isna().any(axis=1).sum()
            if n_missing:
                print(f"Warning: {n_missing} forecast rows had no matching future-exogenous "
                      f"data in X_test — filled with 0.")
                futr_df[self.futr_exog_list] = futr_df[self.futr_exog_list].fillna(0)

        forecast = self.nf.predict(futr_df=futr_df) if futr_df is not None else self.nf.predict()
        forecast['ds'] = pd.to_datetime(forecast['ds'])

        forecast = forecast.rename(columns={self.model.__class__.__name__: 'y_pred'})
        result = test_df.merge(forecast, on=['unique_id', 'ds'], how='left')

        return pd.Series(result['y_pred'].fillna(0).values, index=X_test.index)


## `FeatureEngineer` — calendar & lag features
Adds the countdown/window features you asked for (`days_to_thanksgiving`, `days_to_christmas`, `is_thanksgiving_week`, `is_pre_christmas_week`) alongside the existing calendar and holiday flags. `PrevYearSales` stays as a **historical-only** feature — it references realized sales, but for horizon steps beyond week 52 the year-ago date falls inside the forecast window itself, so it isn't safe to treat as fully future-known.

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin


def _nth_weekday_of_month(year, month, weekday, n):
    """weekday: Monday=0 ... Sunday=6. Returns the date of the n-th such weekday in the month."""
    first = pd.Timestamp(year=year, month=month, day=1)
    days_ahead = (weekday - first.weekday()) % 7
    first_occurrence = first + pd.Timedelta(days=days_ahead)
    return first_occurrence + pd.Timedelta(weeks=n - 1)


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.holidays = {
            'IsSuperBowl': ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'],
            'IsLaborDay': ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'],
            'IsThanksgiving': ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'],
            'IsChristmas': ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']
        }

    def fit(self, X, y=None):
        X = X.copy()
        X['Date'] = pd.to_datetime(X['Date'])
        X['Weekly_Sales'] = y.values
        X['Year'] = X['Date'].dt.year
        X['Week'] = X['Date'].dt.isocalendar().week.astype(int)
        self.full_data_ = X[['Store', 'Dept', 'Date', 'Year', 'Week', 'Weekly_Sales']].copy()
        return self

    def transform(self, X):
        X = X.copy()
        X['Date'] = pd.to_datetime(X['Date'])
        X['Year'] = X['Date'].dt.year
        X['Month'] = X['Date'].dt.month
        X['Week'] = X['Date'].dt.isocalendar().week.astype(int)
        X['Day'] = X['Date'].dt.day
        X['DayOfWeek'] = X['Date'].dt.dayofweek

        for k in range(1, 3):
            X[f'Week_sin_{k}'] = np.sin(2 * np.pi * k * X['Week'] / 52)
            X[f'Week_cos_{k}'] = np.cos(2 * np.pi * k * X['Week'] / 52)

        X['DayOfYear'] = X['Date'].dt.dayofyear
        for k in range(1, 3):
            X[f'DayOfYear_sin_{k}'] = np.sin(2 * np.pi * k * X['DayOfYear'] / 365)
            X[f'DayOfYear_cos_{k}'] = np.cos(2 * np.pi * k * X['DayOfYear'] / 365)

        # Thanksgiving / Christmas countdowns — known for any date, computed generically
        # (4th Thursday of November, Dec 25) instead of a hardcoded year lookup.
        unique_years = X['Year'].unique()
        thanksgiving_by_year = {int(y): _nth_weekday_of_month(int(y), 11, 3, 4) for y in unique_years}
        christmas_by_year = {int(y): pd.Timestamp(year=int(y), month=12, day=25) for y in unique_years}

        X['days_to_thanksgiving'] = (X['Year'].map(thanksgiving_by_year) - X['Date']).dt.days
        X['days_to_christmas'] = (X['Year'].map(christmas_by_year) - X['Date']).dt.days
        X['is_thanksgiving_week'] = (X['days_to_thanksgiving'].abs() <= 3).astype(int)
        X['is_pre_christmas_week'] = ((X['days_to_christmas'] > 0) & (X['days_to_christmas'] <= 10)).astype(int)

        for holiday_name, date_list in self.holidays.items():
            holiday_dates = pd.to_datetime(date_list)

            X[holiday_name] = X['Date'].isin(holiday_dates).astype(int)
            week_before = holiday_dates - pd.Timedelta(weeks=1)
            week_after = holiday_dates + pd.Timedelta(weeks=1)

            X[f'{holiday_name}Before'] = X['Date'].isin(week_before).astype(int)
            X[f'{holiday_name}After'] = X['Date'].isin(week_after).astype(int)

        base = self.full_data_.copy()
        last_year = base.copy()
        last_year['Year'] += 1
        last_year = last_year.rename(columns={'Weekly_Sales': 'Sales_LastYear'})

        df = X.merge(
            last_year[['Store', 'Dept', 'Year', 'Week', 'Sales_LastYear']],
            on=['Store', 'Dept', 'Year', 'Week'],
            how='left'
        )

        two_years_ago = base.copy()
        two_years_ago['Year'] += 2
        two_years_ago = two_years_ago.rename(columns={'Weekly_Sales': 'Sales_TwoYearsAgo'})

        df = df.merge(
            two_years_ago[['Store', 'Dept', 'Year', 'Week', 'Sales_TwoYearsAgo']],
            on=['Store', 'Dept', 'Year', 'Week'],
            how='left'
        )

        df['PrevYearSales'] = df['Sales_LastYear'].fillna(df['Sales_TwoYearsAgo'])
        df.drop(columns=['Sales_LastYear', 'Sales_TwoYearsAgo'], inplace=True)
        df['PrevYearSales'] = df['PrevYearSales'].fillna(0)

        return df


In [9]:
def compute_wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


## GPU memory cleanup
Every sweep loop below trains a fresh `TFT` + `NeuralForecast`/PyTorch Lightning `Trainer` each iteration. Python's garbage collector doesn't always reclaim the previous iteration's CUDA tensors immediately when `model`/`tft_model` go out of scope, so allocations pile up across a loop and can OOM by the 3rd or 4th trial even though any single run fits comfortably in memory. `free_gpu_memory()` is called at the end of every loop body below to force that release.

In [10]:
import gc

def free_gpu_memory(*objs):
    """Delete references and force CUDA to release freed memory before the next fit()."""
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


## Covariate lists

| Type | Columns | Why |
|---|---|---|
| Future (`futr_exog_list`) | holiday flags, `days_to_thanksgiving`, `days_to_christmas`, `is_thanksgiving_week`, `is_pre_christmas_week`, `Temperature`, `Fuel_Price`, `CPI`, `Unemployment`, `MarkDown1`-`5`, calendar sin/cos | known in advance — Walmart's `features.csv` extends across the test period |
| Historical (`hist_exog_list`) | `PrevYearSales` | derived from realized sales; not fully known across a 60-week horizon |
| Static (`stat_exog_list`) | `Store`, `Dept`, `Type`, `Size` | fixed per series, standardized before entering the model |

Yes — using these should help. TFT was built for exactly this: its variable-selection networks learn which covariates matter, and static + future covariates let it distinguish, say, a small Store B electronics department the week before Christmas from a large Store A grocery department in a normal week — signal PatchTST-style univariate models can't see at all.

In [11]:
FUTR_EXOG = [
    'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas',
    'IsSuperBowlBefore', 'IsSuperBowlAfter',
    'IsLaborDayBefore', 'IsLaborDayAfter',
    'IsThanksgivingBefore', 'IsThanksgivingAfter',
    'IsChristmasBefore', 'IsChristmasAfter',
    'days_to_thanksgiving', 'days_to_christmas',
    'is_thanksgiving_week', 'is_pre_christmas_week',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Week_sin_1', 'Week_cos_1', 'Week_sin_2', 'Week_cos_2',
    'DayOfYear_sin_1', 'DayOfYear_cos_1', 'DayOfYear_sin_2', 'DayOfYear_cos_2',
]

HIST_EXOG = ['PrevYearSales']

STAT_EXOG = ['Store', 'Dept', 'Type', 'Size']


## Baseline TFT (target-only, no covariates)
Kept as a reference point to measure how much the covariates below actually help.

In [12]:
from neuralforecast.models import TFT
from sklearn.pipeline import Pipeline

model = TFT(
    h=60,
    input_size=70,
    hidden_size=128,
    n_head=4,
    dropout=0.1,
    attn_dropout=0.0,
    max_steps=100,
    learning_rate=5e-4,
    enable_progress_bar=True,
    start_padding_enabled=True,
    windows_batch_size=256,
    inference_windows_batch_size=256,
)
tft_model = NeuralForecastWrapper(model=model)

tft_model.fit(X_train, y_train)

y_pred = tft_model.predict(X_valid)

wmae_baseline = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
print(f"WMAE (no covariates): {wmae_baseline:.2f}")


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 512    | trai

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE (no covariates): 3262.89


## Hyperparameter search (target-only)

In [13]:
best_wmae = float('inf')
best_param = None

for param in [52, 104]:
    print(f"\ninput_size = {param}")

    model = TFT(
        h=60,
        hidden_size=128,
        n_head=4,
        batch_size=32,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        input_size=param
    )

    tft_model = NeuralForecastWrapper(model=model)
    tft_model.fit(X_train, y_train)

    y_pred = tft_model.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model)

print(f"\nBest input_size: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



input_size = 52


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 512    | train
7 | temporal_encoder        | TemporalCovariateEn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 3309.86

Best input_size: 52 (WMAE: 3207.61)


In [14]:
best_wmae = float('inf')
best_param = None

for param in [64, 128, 256]:
    print(f"\nhidden_size = {param}")

    model = TFT(
        h=60,
        input_size=52,
        n_head=4,
        batch_size=32,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hidden_size=param
    )

    tft_model = NeuralForecastWrapper(model=model)
    tft_model.fit(X_train, y_train)

    y_pred = tft_model.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model)

print(f"\nBest hidden_size: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



hidden_size = 64


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 256    | train
7 | temporal_encoder        | TemporalCovariateEn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 3197.15

Best hidden_size: 64 (WMAE: 3185.17)


In [15]:
best_wmae = float('inf')
best_param = None

for param in [2, 4, 8]:
    print(f"\nn_head = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        batch_size=32,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        n_head=param
    )

    tft_model = NeuralForecastWrapper(model=model)
    tft_model.fit(X_train, y_train)

    y_pred = tft_model.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model)

print(f"\nBest n_head: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



n_head = 2


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 512    | train
7 | temporal_encoder        | TemporalCovariateEn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 3239.75

Best n_head: 4 (WMAE: 3207.61)


In [16]:
best_wmae = float('inf')
best_param = None

for param in [16, 32, 64, 128]:
    print(f"\nbatch_size = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        n_head=4,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        batch_size=param
    )

    tft_model = NeuralForecastWrapper(model=model)
    tft_model.fit(X_train, y_train)

    y_pred = tft_model.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model)

print(f"\nBest batch_size: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



batch_size = 16


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 512    | train
7 | temporal_encoder        | TemporalCovariateEn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 3200.31

Best batch_size: 64 (WMAE: 3159.30)


In [17]:
best_wmae = float('inf')
best_param = None

for param in [0.0001, 0.0005, 0.001, 0.005]:
    print(f"\nlearning_rate = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        n_head=4,
        batch_size=32,
        dropout=0.1,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        learning_rate=param
    )

    tft_model = NeuralForecastWrapper(model=model)
    tft_model.fit(X_train, y_train)

    y_pred = tft_model.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model)

print(f"\nBest learning_rate: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



learning_rate = 0.0001


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 512    | train
7 | temporal_encoder        | TemporalCovariateEn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 3205.36

Best learning_rate: 0.0001 (WMAE: 3202.41)


In [18]:
best_wmae = float('inf')
best_param = None

for param in [0.0, 0.1, 0.2]:
    print(f"\ndropout = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        n_head=4,
        batch_size=32,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        dropout=param
    )

    tft_model = NeuralForecastWrapper(model=model)
    tft_model.fit(X_train, y_train)

    y_pred = tft_model.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model)

print(f"\nBest dropout: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



dropout = 0.0


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 512    | train
7 | temporal_encoder        | TemporalCovariateEn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 3216.32

Best dropout: 0.1 (WMAE: 3207.61)


## Adding future & static covariates
Same sweeps, run through a `Pipeline` with `FeatureEngineer`, and both `TFT` and `NeuralForecastWrapper` given `hist_exog_list` / `futr_exog_list` / `stat_exog_list`.

In [19]:
feature_engineer = FeatureEngineer()

model = TFT(
    h=60,
    input_size=52,
    hidden_size=128,
    n_head=4,
    batch_size=32,
    dropout=0.1,
    attn_dropout=0.0,
    max_steps=100,
    learning_rate=5e-4,
    enable_progress_bar=True,
    start_padding_enabled=True,
    windows_batch_size=256,
    inference_windows_batch_size=256,
    hist_exog_list=HIST_EXOG,
    futr_exog_list=FUTR_EXOG,
    stat_exog_list=STAT_EXOG,
)

tft_model = NeuralForecastWrapper(
    model=model,
    hist_exog_list=HIST_EXOG,
    futr_exog_list=FUTR_EXOG,
    stat_exog_list=STAT_EXOG,
)

pipeline = Pipeline([
    ('feature_engineer', feature_engineer),
    ('model', tft_model)
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_valid)

wmae_covariates = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
print(f"WMAE (with covariates): {wmae_covariates:.2f}")
print(f"WMAE (no covariates):   {wmae_baseline:.2f}")

free_gpu_memory(model, tft_model)


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE (with covariates): 4488.27
WMAE (no covariates):   3262.89


## Hyperparameter search (with covariates)

In [20]:
feature_engineer = FeatureEngineer()

best_wmae = float('inf')
best_param = None

for param in [52, 104]:
    print(f"\ninput_size = {param}")

    model = TFT(
        h=60,
        hidden_size=128,
        n_head=4,
        batch_size=32,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        input_size=param
    )

    tft_model = NeuralForecastWrapper(
        model=model,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
    )

    pipeline = Pipeline([
        ('feature_engineer', feature_engineer),
        ('model', tft_model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model, pipeline)

print(f"\nBest input_size: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



input_size = 52


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2 K | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE: 4367.25

Best input_size: 104 (WMAE: 4367.25)


In [21]:
feature_engineer = FeatureEngineer()

best_wmae = float('inf')
best_param = None

for param in [64, 128, 256]:
    print(f"\nhidden_size = {param}")

    model = TFT(
        h=60,
        input_size=52,
        n_head=4,
        batch_size=32,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        hidden_size=param
    )

    tft_model = NeuralForecastWrapper(
        model=model,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
    )

    pipeline = Pipeline([
        ('feature_engineer', feature_engineer),
        ('model', tft_model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model, pipeline)

print(f"\nBest hidden_size: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



hidden_size = 64


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 5.1 K  | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE: 3678.51

Best hidden_size: 64 (WMAE: 3159.44)


In [22]:
feature_engineer = FeatureEngineer()

best_wmae = float('inf')
best_param = None

for param in [2, 4, 8]:
    print(f"\nn_head = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        batch_size=32,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        n_head=param
    )

    tft_model = NeuralForecastWrapper(
        model=model,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
    )

    pipeline = Pipeline([
        ('feature_engineer', feature_engineer),
        ('model', tft_model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model, pipeline)

print(f"\nBest n_head: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



n_head = 2


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2 K | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE: 3143.22

Best n_head: 8 (WMAE: 3143.22)


In [23]:
feature_engineer = FeatureEngineer()

best_wmae = float('inf')
best_param = None

for param in [16, 32, 64, 128]:
    print(f"\nbatch_size = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        n_head=4,
        dropout=0.1,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        batch_size=param
    )

    tft_model = NeuralForecastWrapper(
        model=model,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
    )

    pipeline = Pipeline([
        ('feature_engineer', feature_engineer),
        ('model', tft_model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model, pipeline)

print(f"\nBest batch_size: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



batch_size = 16


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2 K | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE: 5655.04

Best batch_size: 16 (WMAE: 4314.27)


In [24]:
feature_engineer = FeatureEngineer()

best_wmae = float('inf')
best_param = None

for param in [0.0001, 0.0005, 0.001, 0.005]:
    print(f"\nlearning_rate = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        n_head=4,
        batch_size=32,
        dropout=0.1,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        learning_rate=param
    )

    tft_model = NeuralForecastWrapper(
        model=model,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
    )

    pipeline = Pipeline([
        ('feature_engineer', feature_engineer),
        ('model', tft_model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model, pipeline)

print(f"\nBest learning_rate: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



learning_rate = 0.0001


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2 K | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE: 3146.17

Best learning_rate: 0.005 (WMAE: 3146.17)


In [25]:
feature_engineer = FeatureEngineer()

best_wmae = float('inf')
best_param = None

for param in [0.0, 0.1, 0.2]:
    print(f"\ndropout = {param}")

    model = TFT(
        h=60,
        input_size=52,
        hidden_size=128,
        n_head=4,
        batch_size=32,
        learning_rate=5e-4,
        attn_dropout=0.0,
        max_steps=100,
        enable_progress_bar=True,
        start_padding_enabled=True,
        windows_batch_size=256,
        inference_windows_batch_size=256,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        dropout=param
    )

    tft_model = NeuralForecastWrapper(
        model=model,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
    )

    pipeline = Pipeline([
        ('feature_engineer', feature_engineer),
        ('model', tft_model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_valid)

    wmae = compute_wmae(y_valid.values, y_pred.values, X_valid['IsHoliday'].values)
    print(f"WMAE: {wmae:.2f}")

    if wmae < best_wmae:
        best_wmae = wmae
        best_param = param

    free_gpu_memory(model, tft_model, pipeline)

print(f"\nBest dropout: {best_param} (WMAE: {best_wmae:.2f})")


INFO:lightning_fabric.utilities.seed:Seed set to 1



dropout = 0.0


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2 K | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


WMAE: 4415.84

Best dropout: 0.0 (WMAE: 4201.44)


## Final fit and Kaggle submission
`test` has to go through the same `Preprocessor` as `train` before it reaches the model — otherwise it has no `Type`, `Size`, `Temperature`, `MarkDown*`, etc., and every covariate above would be silently missing at inference. The original notebook predicted on raw `test`; fixed here.

In [26]:
feature_engineer = FeatureEngineer()

model = TFT(
    h=60,
    input_size=52,
    hidden_size=128,
    n_head=8,
    batch_size=64,
    dropout=0.1,
    attn_dropout=0.0,
    max_steps=5000,
    learning_rate=0.0005,
    grn_activation='ELU',
    enable_progress_bar=True,
    start_padding_enabled=True,
    windows_batch_size=256,
    inference_windows_batch_size=256,
    hist_exog_list=HIST_EXOG,
    futr_exog_list=FUTR_EXOG,
    stat_exog_list=STAT_EXOG,
)

tft_model = NeuralForecastWrapper(
    model=model,
    hist_exog_list=HIST_EXOG,
    futr_exog_list=FUTR_EXOG,
    stat_exog_list=STAT_EXOG,
)

pipeline = Pipeline([
    ('feature_engineer', feature_engineer),
    ('model', tft_model)
])

y = df['Weekly_Sales']
X = df.drop(columns=['Weekly_Sales'])

pipeline.fit(X, y)

test_processed = preprocessor.transform(test)

test_pred = pipeline.predict(test_processed)
submission = test_processed[['Store', 'Dept', 'Date']].copy()
submission['Id'] = (
    submission['Store'].astype(str) + '_' +
    submission['Dept'].astype(str) + '_' +
    submission['Date'].astype(str)
)
submission['Weekly_Sales'] = test_pred.values
submission = submission[['Id', 'Weekly_Sales']]
submission.to_csv('submission_tft.csv', index=False)
submission.head()


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 10.2

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=5000` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


,Id,Weekly_Sales
0,1_1_2012-11-02,25262.285156
1,1_1_2012-11-09,25272.617188
2,1_1_2012-11-16,18779.119141
3,1_1_2012-11-23,23182.763672
4,1_1_2012-11-30,20941.750000


## Log the run to MLflow

In [28]:
with mlflow.start_run(run_name="TFT_Model") as run:
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="TFT_pipeline_model",
        serialization_format="cloudpickle",
        registered_model_name="TFTPipelineModel"
    )

    mlflow.log_params({
        'model': 'TFT',
        'h': 60,
        'input_size': 52,
        'hidden_size': 128,
        'n_head': 8,
        'batch_size': 64,
        'dropout': 0.1,
        'attn_dropout': 0.0,
        'max_steps': 5000,
        'learning_rate': 0.0005,
        'grn_activation': 'ELU',
        'futr_exog_list': FUTR_EXOG,
        'hist_exog_list': HIST_EXOG,
        'stat_exog_list': STAT_EXOG,
    })

    mlflow.log_metric("wmae", wmae_covariates)
    mlflow.log_metric("wmae_baseline_no_covariates", wmae_baseline)


2026/07/25 16:35:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/25 16:36:22 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'TFTPipelineModel'.
2026/07/25 16:36:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: TFTPipelineMod

🏃 View run TFT_Model at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/44/runs/2eda90d6adcb4124a6972a3a6a5681fa
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/44
